# Notebook 06: Financial Commitment, Premium Risk & Retention Priority Analysis

This notebook analyzes how financial commitment, customer activity, premium customer behavior, and relationship strength influence customer retention.

The main goal is to identify high-value customers who may be disengaged, financially important customers who may still churn, and customer segments that should receive higher retention priority.

This notebook supports the following project guideline areas:

- Financial Commitment vs Engagement Analysis
- High-Balance Disengagement Rate
- Salary–Balance Mismatch Detection
- At-Risk Premium Customer Identification
- Credit Card Stickiness Score
- Relationship Strength Assessment
- Retention Priority Scoring

## Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Libraries imported successfully.")

Libraries imported successfully.


## Define Project Paths

In [2]:
# Define project paths
BASE_DIR = Path.cwd().parent

FEATURES_DATA_PATH = BASE_DIR / "data" / "processed" / "european_bank_features.csv"
RETENTION_OUTPUT_DIR = BASE_DIR / "data" / "processed" / "retention_priority_analysis"

print("Base Directory:", BASE_DIR)
print("Feature-Engineered Data Path:", FEATURES_DATA_PATH)
print("Retention Output Directory:", RETENTION_OUTPUT_DIR)

Base Directory: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy
Feature-Engineered Data Path: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\processed\european_bank_features.csv
Retention Output Directory: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\processed\retention_priority_analysis


## Load Feature-Engineered Dataset

In [3]:
# Load feature-engineered master dataset
df = pd.read_csv(FEATURES_DATA_PATH)

# Preview dataset
df.head()

,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Churn_Status,Engagement_Status,Credit_Card_Status,Product_Depth_Segment,Balance_Segment,Salary_Segment,Age_Segment,Tenure_Segment,High_Value_Customer_Flag,High_Value_Customer_Status,High_Value_Disengaged_Flag,High_Value_Disengaged_Status,At_Risk_Premium_Flag,At_Risk_Premium_Status,Salary_Balance_Mismatch_Flag,Salary_Balance_Mismatch_Status,Customer_Risk_Segment,Relationship_Strength_Index,Relationship_Strength_Tier
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,"101,348.88",1,Churned,Active Member,Has Credit Card,Single Product,Zero Balance,High Salary,Mid-Age Customer,New Customer,0,Standard Value Customer,0,Not High-Value Disengaged,0,Not At-Risk Premium,0,No Salary-Balance Mismatch,Standard Monitoring,55,Moderate Relationship
1,2025,15647311,Hill,608,Spain,Female,41,1,"83,807.86",1,0,1,"112,542.58",0,Retained,Active Member,No Credit Card,Single Product,Medium Balance,High Salary,Mid-Age Customer,New Customer,0,Standard Value Customer,0,Not High-Value Disengaged,0,Not At-Risk Premium,0,No Salary-Balance Mismatch,Standard Monitoring,55,Moderate Relationship
2,2025,15619304,Onio,502,France,Female,42,8,"159,660.80",3,1,0,"113,931.57",1,Churned,Inactive Member,Has Credit Card,High Product Load,Premium Balance,High Salary,Mid-Age Customer,Established Relationship,1,High Value Customer,1,High-Value Disengaged,1,At-Risk Premium Customer,0,No Salary-Balance Mismatch,At-Risk Premium,50,Weak Relationship
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,"93,826.63",0,Retained,Inactive Member,No Credit Card,Optimal Product Depth,Zero Balance,Medium Salary,Mid-Age Customer,New Customer,0,Standard Value Customer,0,Not High-Value Disengaged,0,Not At-Risk Premium,0,No Salary-Balance Mismatch,Standard Monitoring,35,Weak Relationship
4,2025,15737888,Mitchell,850,Spain,Female,43,2,"125,510.82",1,1,1,"79,084.10",0,Retained,Active Member,Has Credit Card,Single Product,High Balance,Medium Salary,Mid-Age Customer,New Customer,0,Standard Value Customer,0,Not High-Value Disengaged,0,Not At-Risk Premium,0,No Salary-Balance Mismatch,Standard Monitoring,75,Moderate Relationship


## Dataset Shape and Column Validation

This step confirms that Notebook 06 is using the master feature-engineered dataset exported from Notebook 03.

In [4]:
print(f"Total Rows: {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]:,}")

required_columns = [
    "CustomerId",
    "Balance",
    "EstimatedSalary",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "Exited",
    "Engagement_Status",
    "Credit_Card_Status",
    "Balance_Segment",
    "Salary_Segment",
    "High_Value_Customer_Flag",
    "High_Value_Customer_Status",
    "High_Value_Disengaged_Flag",
    "High_Value_Disengaged_Status",
    "At_Risk_Premium_Flag",
    "At_Risk_Premium_Status",
    "Salary_Balance_Mismatch_Flag",
    "Salary_Balance_Mismatch_Status",
    "Relationship_Strength_Index",
    "Relationship_Strength_Tier",
    "Customer_Risk_Segment"
]

missing_required_columns = [col for col in required_columns if col not in df.columns]

if len(missing_required_columns) == 0:
    print("All required columns are available.")
else:
    print("Missing required columns:")
    print(missing_required_columns)

Total Rows: 10,000
Total Columns: 33
All required columns are available.


## Customer Risk Segment Distribution

This step reviews the behavior-based customer risk segments created during feature engineering.

In [5]:
customer_risk_distribution = (
    df["Customer_Risk_Segment"]
    .value_counts()
    .reset_index()
)

customer_risk_distribution.columns = ["Customer Risk Segment", "Customer Count"]

customer_risk_distribution["Customer %"] = (
    customer_risk_distribution["Customer Count"] / len(df) * 100
).round(2)

customer_risk_distribution

,Customer Risk Segment,Customer Count,Customer %
0,At-Risk Premium,3452,34.52
1,Standard Monitoring,2641,26.41
2,Sticky Active Customer,2446,24.46
3,Inactive Single-Product Risk,1279,12.79
4,High Product Load Risk,182,1.82


## Customer Risk Segment vs Churn Validation

This step validates whether customer risk segments show meaningful churn differences.

In [6]:
customer_risk_churn = (
    df
    .groupby("Customer_Risk_Segment")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Retained_Customers=("Exited", lambda x: (x == 0).sum()),
        Churn_Rate=("Exited", "mean"),
        Avg_Balance=("Balance", "mean"),
        Avg_Salary=("EstimatedSalary", "mean"),
        Avg_Num_Products=("NumOfProducts", "mean"),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean")
    )
    .reset_index()
)

customer_risk_churn["Churn_Rate"] = (
    customer_risk_churn["Churn_Rate"] * 100
).round(2)

customer_risk_churn["Retention_Rate"] = (
    customer_risk_churn["Retained_Customers"] / customer_risk_churn["Customer_Count"] * 100
).round(2)

for col in ["Avg_Balance", "Avg_Salary", "Avg_Num_Products", "Avg_Relationship_Strength"]:
    customer_risk_churn[col] = customer_risk_churn[col].round(2)

customer_risk_churn = customer_risk_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

customer_risk_churn

,Customer_Risk_Segment,Customer_Count,Churned_Customers,Retained_Customers,Churn_Rate,Avg_Balance,Avg_Salary,Avg_Num_Products,Avg_Relationship_Strength,Retention_Rate
1,High Product Load Risk,182,145,37,79.67,"54,044.33","79,915.40",3.16,46.07,20.33
2,Inactive Single-Product Risk,1279,479,800,37.45,"72,971.53","76,797.70",1.00,40.55,62.55
0,At-Risk Premium,3452,912,2540,26.42,"113,780.60","129,135.43",1.33,60.52,73.58
3,Standard Monitoring,2641,365,2276,13.82,"53,801.33","74,446.45",1.50,62.16,86.18
4,Sticky Active Customer,2446,136,2310,5.56,"51,852.94","100,468.10",2.00,86.31,94.44


## High-Balance Disengagement Rate

This KPI measures the share of high-balance customers who are inactive.

This supports the project requirement: High-Balance Disengagement Rate.

In [7]:
# Define high-balance customers using the 75th percentile of Balance
high_balance_threshold = df["Balance"].quantile(0.75)

df["High_Balance_Flag"] = np.where(
    df["Balance"] >= high_balance_threshold,
    1,
    0
)

df["High_Balance_Status"] = df["High_Balance_Flag"].map({
    0: "Not High Balance",
    1: "High Balance Customer"
})

high_balance_customers = df[df["High_Balance_Flag"] == 1]
high_balance_disengaged_customers = high_balance_customers[
    high_balance_customers["IsActiveMember"] == 0
]

high_balance_disengagement_rate = (
    len(high_balance_disengaged_customers) / len(high_balance_customers) * 100
)

print(f"High Balance Threshold Used: {high_balance_threshold:,.2f}")
print(f"High Balance Customers: {len(high_balance_customers):,}")
print(f"High Balance Disengaged Customers: {len(high_balance_disengaged_customers):,}")
print(f"High-Balance Disengagement Rate: {high_balance_disengagement_rate:.2f}%")

High Balance Threshold Used: 127,644.24
High Balance Customers: 2,500
High Balance Disengaged Customers: 1,247
High-Balance Disengagement Rate: 49.88%


## High-Balance Engagement vs Churn Analysis

This step analyzes whether high-balance inactive customers show higher churn risk.

In [8]:
high_balance_engagement_churn = (
    df
    .groupby(["High_Balance_Status", "Engagement_Status"])
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Retained_Customers=("Exited", lambda x: (x == 0).sum()),
        Churn_Rate=("Exited", "mean"),
        Avg_Balance=("Balance", "mean"),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean")
    )
    .reset_index()
)

high_balance_engagement_churn["Churn_Rate"] = (
    high_balance_engagement_churn["Churn_Rate"] * 100
).round(2)

high_balance_engagement_churn["Retention_Rate"] = (
    high_balance_engagement_churn["Retained_Customers"] /
    high_balance_engagement_churn["Customer_Count"] * 100
).round(2)

high_balance_engagement_churn["Avg_Balance"] = (
    high_balance_engagement_churn["Avg_Balance"]
).round(2)

high_balance_engagement_churn["Avg_Relationship_Strength"] = (
    high_balance_engagement_churn["Avg_Relationship_Strength"]
).round(2)

high_balance_engagement_churn = high_balance_engagement_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

high_balance_engagement_churn

,High_Balance_Status,Engagement_Status,Customer_Count,Churned_Customers,Retained_Customers,Churn_Rate,Avg_Balance,Avg_Relationship_Strength,Retention_Rate
1,High Balance Customer,Inactive Member,1247,380,867,30.47,"148,857.68",54.86,69.53
3,Not High Balance,Inactive Member,3602,922,2680,25.60,"52,304.02",46.85,74.40
0,High Balance Customer,Active Member,1253,212,1041,16.92,"149,243.04",84.63,83.08
2,Not High Balance,Active Member,3898,523,3375,13.42,"52,291.63",77.28,86.58


## Credit Card Stickiness Score

This KPI compares retention between customers with and without credit cards.

Credit card ownership should be interpreted carefully because earlier validation showed only a small difference in churn rates.

In [9]:
credit_card_stickiness = (
    df
    .groupby("Credit_Card_Status")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Retained_Customers=("Exited", lambda x: (x == 0).sum()),
        Churn_Rate=("Exited", "mean"),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean")
    )
    .reset_index()
)

credit_card_stickiness["Churn_Rate"] = (
    credit_card_stickiness["Churn_Rate"] * 100
).round(2)

credit_card_stickiness["Retention_Rate"] = (
    credit_card_stickiness["Retained_Customers"] /
    credit_card_stickiness["Customer_Count"] * 100
).round(2)

credit_card_stickiness["Avg_Relationship_Strength"] = (
    credit_card_stickiness["Avg_Relationship_Strength"]
).round(2)

credit_card_retention = credit_card_stickiness.loc[
    credit_card_stickiness["Credit_Card_Status"] == "Has Credit Card",
    "Retention_Rate"
].iloc[0]

no_credit_card_retention = credit_card_stickiness.loc[
    credit_card_stickiness["Credit_Card_Status"] == "No Credit Card",
    "Retention_Rate"
].iloc[0]

credit_card_stickiness_score = credit_card_retention - no_credit_card_retention

print(f"Credit Card Customer Retention Rate: {credit_card_retention:.2f}%")
print(f"No Credit Card Customer Retention Rate: {no_credit_card_retention:.2f}%")
print(f"Credit Card Stickiness Score: {credit_card_stickiness_score:.2f} percentage points")

credit_card_stickiness

Credit Card Customer Retention Rate: 79.82%
No Credit Card Customer Retention Rate: 79.19%
Credit Card Stickiness Score: 0.63 percentage points


,Credit_Card_Status,Customer_Count,Churned_Customers,Retained_Customers,Churn_Rate,Avg_Relationship_Strength,Retention_Rate
0,Has Credit Card,7055,1424,5631,20.18,67.29,79.82
1,No Credit Card,2945,613,2332,20.81,57.62,79.19


## Salary-Balance Mismatch Distribution

This step reviews customers whose estimated salary is high but account balance is unusually low.  
This can indicate weak financial commitment, low wallet share, or possible retention risk.

In [10]:
# Analyze salary-balance mismatch status distribution
salary_balance_mismatch_distribution = (
    df["Salary_Balance_Mismatch_Status"]
    .value_counts()
    .reset_index()
)

salary_balance_mismatch_distribution.columns = [
    "Salary Balance Mismatch Status",
    "Customer Count"
]

salary_balance_mismatch_distribution["Customer %"] = (
    salary_balance_mismatch_distribution["Customer Count"] / len(df) * 100
).round(2)

salary_balance_mismatch_distribution

,Salary Balance Mismatch Status,Customer Count,Customer %
0,No Salary-Balance Mismatch,9109,91.09
1,Salary-Balance Mismatch,891,8.91


## Salary-Balance Mismatch vs Churn Validation

This step checks whether salary-balance mismatch customers show higher or lower churn behavior compared with normal customers.

In [11]:
# Analyze churn behavior by salary-balance mismatch status
salary_balance_mismatch_churn = (
    df
    .groupby("Salary_Balance_Mismatch_Status")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Retained_Customers=("Exited", lambda x: (x == 0).sum()),
        Avg_Balance=("Balance", "mean"),
        Avg_Salary=("EstimatedSalary", "mean"),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean")
    )
    .reset_index()
)

salary_balance_mismatch_churn["Churn_Rate"] = (
    salary_balance_mismatch_churn["Churned_Customers"] /
    salary_balance_mismatch_churn["Customer_Count"] * 100
).round(2)

salary_balance_mismatch_churn["Retention_Rate"] = (
    salary_balance_mismatch_churn["Retained_Customers"] /
    salary_balance_mismatch_churn["Customer_Count"] * 100
).round(2)

for col in ["Avg_Balance", "Avg_Salary", "Avg_Relationship_Strength"]:
    salary_balance_mismatch_churn[col] = salary_balance_mismatch_churn[col].round(2)

salary_balance_mismatch_churn = salary_balance_mismatch_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

salary_balance_mismatch_churn

,Salary_Balance_Mismatch_Status,Customer_Count,Churned_Customers,Retained_Customers,Avg_Balance,Avg_Salary,Avg_Relationship_Strength,Churn_Rate,Retention_Rate
0,No Salary-Balance Mismatch,9109,1905,7204,"83,967.38","92,806.54",65.00,20.91,79.09
1,Salary-Balance Mismatch,891,132,759,0.00,"174,553.97",58.74,14.81,85.19


## At-Risk Premium Customer Distribution

This step reviews customers classified as premium risk customers based on high salary and low/weak balance behavior.

In [12]:
# Analyze at-risk premium customer distribution
at_risk_premium_distribution = (
    df["At_Risk_Premium_Status"]
    .value_counts()
    .reset_index()
)

at_risk_premium_distribution.columns = [
    "At-Risk Premium Status",
    "Customer Count"
]

at_risk_premium_distribution["Customer %"] = (
    at_risk_premium_distribution["Customer Count"] / len(df) * 100
).round(2)

at_risk_premium_distribution

,At-Risk Premium Status,Customer Count,Customer %
0,Not At-Risk Premium,6548,65.48
1,At-Risk Premium Customer,3452,34.52


## At-Risk Premium vs Churn Validation

This step checks whether at-risk premium customers show higher churn compared with standard premium/non-risk customers.

In [13]:
# Analyze churn behavior by at-risk premium status
at_risk_premium_churn = (
    df
    .groupby("At_Risk_Premium_Status")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Retained_Customers=("Exited", lambda x: (x == 0).sum()),
        Avg_Balance=("Balance", "mean"),
        Avg_Salary=("EstimatedSalary", "mean"),
        Avg_Num_Products=("NumOfProducts", "mean"),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean")
    )
    .reset_index()
)

at_risk_premium_churn["Churn_Rate"] = (
    at_risk_premium_churn["Churned_Customers"] /
    at_risk_premium_churn["Customer_Count"] * 100
).round(2)

at_risk_premium_churn["Retention_Rate"] = (
    at_risk_premium_churn["Retained_Customers"] /
    at_risk_premium_churn["Customer_Count"] * 100
).round(2)

for col in ["Avg_Balance", "Avg_Salary", "Avg_Num_Products", "Avg_Relationship_Strength"]:
    at_risk_premium_churn[col] = at_risk_premium_churn[col].round(2)

at_risk_premium_churn = at_risk_premium_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

at_risk_premium_churn

,At_Risk_Premium_Status,Customer_Count,Churned_Customers,Retained_Customers,Avg_Balance,Avg_Salary,Avg_Num_Products,Avg_Relationship_Strength,Churn_Rate,Retention_Rate
0,At-Risk Premium Customer,3452,912,2540,"113,780.60","129,135.43",1.33,60.52,26.42,73.58
1,Not At-Risk Premium,6548,1125,5423,"56,824.72","84,778.09",1.63,66.51,17.18,82.82


## High-Balance Disengaged Customer Deep Dive

This step focuses on customers who hold high balances but are inactive.  
These customers are financially valuable but may require retention attention due to disengagement.

In [14]:
# Create high-balance disengaged customer subset
high_balance_disengaged_customers = df[
    (df["High_Balance_Flag"] == 1) &
    (df["IsActiveMember"] == 0)
].copy()

high_balance_disengaged_summary = {
    "High Balance Disengaged Customers": len(high_balance_disengaged_customers),
    "Churned Customers": int(high_balance_disengaged_customers["Exited"].sum()),
    "Retained Customers": int((high_balance_disengaged_customers["Exited"] == 0).sum()),
    "Churn Rate": round(high_balance_disengaged_customers["Exited"].mean() * 100, 2),
    "Retention Rate": round((1 - high_balance_disengaged_customers["Exited"].mean()) * 100, 2),
    "Average Balance": round(high_balance_disengaged_customers["Balance"].mean(), 2),
    "Average Salary": round(high_balance_disengaged_customers["EstimatedSalary"].mean(), 2),
    "Average Relationship Strength": round(high_balance_disengaged_customers["Relationship_Strength_Index"].mean(), 2)
}

high_balance_disengaged_summary

{'High Balance Disengaged Customers': 1247,
 'Churned Customers': 380,
 'Retained Customers': 867,
 'Churn Rate': np.float64(30.47),
 'Retention Rate': np.float64(69.53),
 'Average Balance': np.float64(148857.68),
 'Average Salary': np.float64(101338.72),
 'Average Relationship Strength': np.float64(54.86)}

## High-Balance Disengaged Segment Table

This table prepares high-balance disengaged customer information for dashboard prioritization.

In [15]:
# Prepare high-balance disengaged customer table
high_balance_disengaged_table = high_balance_disengaged_customers[
    [
        "CustomerId",
        "Geography",
        "Gender",
        "Age",
        "Tenure",
        "Balance",
        "EstimatedSalary",
        "NumOfProducts",
        "Credit_Card_Status",
        "Engagement_Status",
        "Product_Depth_Segment",
        "Customer_Risk_Segment",
        "Relationship_Strength_Index",
        "Exited"
    ]
].copy()

high_balance_disengaged_table["Churn_Status"] = high_balance_disengaged_table["Exited"].map({
    0: "Retained",
    1: "Churned"
})

high_balance_disengaged_table = high_balance_disengaged_table.drop(columns=["Exited"])

high_balance_disengaged_table = high_balance_disengaged_table.sort_values(
    by=["Relationship_Strength_Index", "Balance"],
    ascending=[True, False]
)

high_balance_disengaged_table.head(10)

,CustomerId,Geography,Gender,Age,Tenure,Balance,EstimatedSalary,NumOfProducts,Credit_Card_Status,Engagement_Status,Product_Depth_Segment,Customer_Risk_Segment,Relationship_Strength_Index,Churn_Status
2658,15815490,Germany,Male,40,2,"164,948.98","177,028.00",3,No Credit Card,Inactive Member,High Product Load,At-Risk Premium,30,Churned
4013,15653776,Germany,Female,57,1,"162,082.31","27,145.73",4,No Credit Card,Inactive Member,High Product Load,At-Risk Premium,30,Churned
4779,15711843,Germany,Male,40,1,"147,856.82","107,961.11",3,No Credit Card,Inactive Member,High Product Load,At-Risk Premium,30,Churned
6295,15621550,Spain,Female,50,1,"140,292.58","69,531.22",3,No Credit Card,Inactive Member,High Product Load,At-Risk Premium,30,Churned
5871,15709920,France,Female,33,2,"208,165.53","50,774.81",1,No Credit Card,Inactive Member,Single Product,At-Risk Premium,35,Churned
980,15624729,France,Male,27,0,"197,041.80","151,912.49",1,No Credit Card,Inactive Member,Single Product,At-Risk Premium,35,Retained
5604,15804721,France,Male,49,0,"191,808.73","97,640.20",1,No Credit Card,Inactive Member,Single Product,At-Risk Premium,35,Retained
2207,15731507,France,Female,33,1,"188,285.68","58,363.94",1,No Credit Card,Inactive Member,Single Product,At-Risk Premium,35,Retained
6567,15593425,Spain,Female,54,1,"187,997.15","111,442.71",1,No Credit Card,Inactive Member,Single Product,At-Risk Premium,35,Churned
841,15737792,France,Female,31,1,"186,796.37","178,252.63",1,No Credit Card,Inactive Member,Single Product,At-Risk Premium,35,Retained


## Retention Priority Segment Creation

This step creates a retention priority label using customer risk, financial value, engagement behavior, and relationship strength.

In [16]:
# Create retention priority segment
def assign_retention_priority(row):
    if row["Exited"] == 1:
        return "Already Churned"
    elif (
        row["High_Value_Disengaged_Flag"] == 1 or
        row["Customer_Risk_Segment"] == "High Product Load Risk"
    ):
        return "Critical Priority"
    elif (
        row["At_Risk_Premium_Flag"] == 1 or
        row["Salary_Balance_Mismatch_Flag"] == 1 or
        row["Customer_Risk_Segment"] == "Inactive Single-Product Risk"
    ):
        return "High Priority"
    elif row["Relationship_Strength_Index"] < 50:
        return "Medium Priority"
    else:
        return "Low Priority"

df["Retention_Priority_Segment"] = df.apply(assign_retention_priority, axis=1)

retention_priority_distribution = (
    df["Retention_Priority_Segment"]
    .value_counts()
    .reset_index()
)

retention_priority_distribution.columns = [
    "Retention Priority Segment",
    "Customer Count"
]

retention_priority_distribution["Customer %"] = (
    retention_priority_distribution["Customer Count"] / len(df) * 100
).round(2)

retention_priority_distribution

,Retention Priority Segment,Customer Count,Customer %
0,Low Priority,3873,38.73
1,High Priority,2143,21.43
2,Already Churned,2037,20.37
3,Critical Priority,1565,15.65
4,Medium Priority,382,3.82


## Retention Priority vs Churn Validation

This step validates whether the created retention priority segments correctly separate customer risk groups.

In [17]:
# Analyze churn behavior by retention priority segment
retention_priority_churn = (
    df
    .groupby("Retention_Priority_Segment")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Retained_Customers=("Exited", lambda x: (x == 0).sum()),
        Avg_Balance=("Balance", "mean"),
        Avg_Salary=("EstimatedSalary", "mean"),
        Avg_Num_Products=("NumOfProducts", "mean"),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean")
    )
    .reset_index()
)

retention_priority_churn["Churn_Rate"] = (
    retention_priority_churn["Churned_Customers"] /
    retention_priority_churn["Customer_Count"] * 100
).round(2)

retention_priority_churn["Retention_Rate"] = (
    retention_priority_churn["Retained_Customers"] /
    retention_priority_churn["Customer_Count"] * 100
).round(2)

for col in ["Avg_Balance", "Avg_Salary", "Avg_Num_Products", "Avg_Relationship_Strength"]:
    retention_priority_churn[col] = retention_priority_churn[col].round(2)

retention_priority_churn = retention_priority_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

retention_priority_churn

,Retention_Priority_Segment,Customer_Count,Churned_Customers,Retained_Customers,Avg_Balance,Avg_Salary,Avg_Num_Products,Avg_Relationship_Strength,Churn_Rate,Retention_Rate
0,Already Churned,2037,2037,0,"91,108.54","101,465.68",1.48,55.68,100.00,0.00
1,Critical Priority,1565,0,1565,"102,362.71","131,711.50",1.52,52.88,0.00,100.00
2,High Priority,2143,0,2143,"87,698.65","113,826.85",1.16,63.26,0.00,100.00
3,Low Priority,3873,0,3873,"59,476.03","81,471.58",1.73,76.56,0.00,100.00
4,Medium Priority,382,0,382,"2,053.11","74,916.07",1.96,42.30,0.00,100.00


## Financial Commitment KPI Summary

This step prepares dashboard-ready KPIs for financial commitment, premium risk, and retention priority analysis.

In [18]:
# Calculate financial commitment KPIs
salary_balance_mismatch_rate = (
    df["Salary_Balance_Mismatch_Flag"].mean() * 100
)

at_risk_premium_rate = (
    df["At_Risk_Premium_Flag"].mean() * 100
)

high_balance_disengagement_rate = (
    len(high_balance_disengaged_customers) /
    len(df[df["High_Balance_Flag"] == 1]) * 100
)

critical_priority_customers = (
    df[df["Retention_Priority_Segment"] == "Critical Priority"].shape[0]
)

high_priority_customers = (
    df[df["Retention_Priority_Segment"] == "High Priority"].shape[0]
)

credit_card_retention_rate = credit_card_retention

no_credit_card_retention_rate = no_credit_card_retention

credit_card_stickiness_score = credit_card_stickiness_score

financial_commitment_kpi_summary = pd.DataFrame({
    "KPI": [
        "Salary-Balance Mismatch Rate",
        "At-Risk Premium Customer Rate",
        "High-Balance Disengagement Rate",
        "Critical Priority Customers",
        "High Priority Customers",
        "Credit Card Customer Retention Rate",
        "No Credit Card Customer Retention Rate",
        "Credit Card Stickiness Score"
    ],
    "Value": [
        f"{salary_balance_mismatch_rate:.2f}%",
        f"{at_risk_premium_rate:.2f}%",
        f"{high_balance_disengagement_rate:.2f}%",
        f"{critical_priority_customers:,}",
        f"{high_priority_customers:,}",
        f"{credit_card_retention_rate:.2f}%",
        f"{no_credit_card_retention_rate:.2f}%",
        f"{credit_card_stickiness_score:.2f} percentage points"
    ],
    "Business Meaning": [
        "Share of customers with high salary but weak balance commitment",
        "Share of customers showing premium financial risk behavior",
        "Inactive share among high-balance customers",
        "Customers requiring immediate retention attention",
        "Customers requiring proactive retention monitoring",
        "Retention rate among credit card customers",
        "Retention rate among non-credit card customers",
        "Retention lift from credit card ownership"
    ]
})

financial_commitment_kpi_summary

,KPI,Value,Business Meaning
0,Salary-Balance Mismatch Rate,8.91%,Share of customers with high salary but weak b...
1,At-Risk Premium Customer Rate,34.52%,Share of customers showing premium financial r...
2,High-Balance Disengagement Rate,49.88%,Inactive share among high-balance customers
3,Critical Priority Customers,"1,565",Customers requiring immediate retention attention
4,High Priority Customers,"2,143",Customers requiring proactive retention monito...
5,Credit Card Customer Retention Rate,79.82%,Retention rate among credit card customers
6,No Credit Card Customer Retention Rate,79.19%,Retention rate among non-credit card customers
7,Credit Card Stickiness Score,0.63 percentage points,Retention lift from credit card ownership


## Financial Commitment Insight Table

This table converts the major findings from Notebook 06 into dashboard-ready insight messages.

In [19]:
# Prepare financial commitment insight table
financial_commitment_insight_table = pd.DataFrame({
    "Insight Area": [
        "High-Balance Disengagement",
        "Salary-Balance Mismatch",
        "At-Risk Premium Customers",
        "Credit Card Stickiness",
        "Retention Priority",
        "Financial Commitment Risk"
    ],
    "Key Finding": [
        "High-balance inactive customers represent valuable but potentially vulnerable customers.",
        "Salary-balance mismatch identifies customers with high earning potential but weak balance commitment.",
        "At-risk premium customers require proactive monitoring due to financial value and risk behavior.",
        "Credit card ownership provides only a small retention difference and should be combined with other signals.",
        "Retention priority segments help separate customers by action urgency.",
        "Financial value alone does not guarantee retention without engagement and relationship strength."
    ],
    "Metric Evidence": [
        f"High-balance disengagement rate: {high_balance_disengagement_rate:.2f}%",
        f"Salary-balance mismatch rate: {salary_balance_mismatch_rate:.2f}%",
        f"At-risk premium customer rate: {at_risk_premium_rate:.2f}%",
        f"Credit card stickiness score: {credit_card_stickiness_score:.2f} percentage points",
        f"Critical priority customers: {critical_priority_customers:,}",
        "Balance, salary, engagement, product usage, and RSI combined"
    ],
    "Dashboard Use": [
        "High-value disengaged customer detector",
        "Salary-balance mismatch warning card",
        "Premium customer risk panel",
        "Credit card stickiness KPI",
        "Retention priority scoring panel",
        "Customer risk segmentation table"
    ]
})

financial_commitment_insight_table

,Insight Area,Key Finding,Metric Evidence,Dashboard Use
0,High-Balance Disengagement,High-balance inactive customers represent valu...,High-balance disengagement rate: 49.88%,High-value disengaged customer detector
1,Salary-Balance Mismatch,Salary-balance mismatch identifies customers w...,Salary-balance mismatch rate: 8.91%,Salary-balance mismatch warning card
2,At-Risk Premium Customers,At-risk premium customers require proactive mo...,At-risk premium customer rate: 34.52%,Premium customer risk panel
3,Credit Card Stickiness,Credit card ownership provides only a small re...,Credit card stickiness score: 0.63 percentage ...,Credit card stickiness KPI
4,Retention Priority,Retention priority segments help separate cust...,"Critical priority customers: 1,565",Retention priority scoring panel
5,Financial Commitment Risk,Financial value alone does not guarantee reten...,"Balance, salary, engagement, product usage, an...",Customer risk segmentation table


### Export Notebook 06 Analysis Outputs

In [20]:
# Step 6U: Export financial commitment and retention priority analysis outputs

# Create output directory if it does not exist
RETENTION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Export notebook 06 analysis outputs
customer_risk_distribution.to_csv(
    RETENTION_OUTPUT_DIR / "customer_risk_distribution.csv",
    index=False
)

customer_risk_churn.to_csv(
    RETENTION_OUTPUT_DIR / "customer_risk_churn.csv",
    index=False
)

high_balance_engagement_churn.to_csv(
    RETENTION_OUTPUT_DIR / "high_balance_engagement_churn.csv",
    index=False
)

salary_balance_mismatch_distribution.to_csv(
    RETENTION_OUTPUT_DIR / "salary_balance_mismatch_distribution.csv",
    index=False
)

salary_balance_mismatch_churn.to_csv(
    RETENTION_OUTPUT_DIR / "salary_balance_mismatch_churn.csv",
    index=False
)

at_risk_premium_distribution.to_csv(
    RETENTION_OUTPUT_DIR / "at_risk_premium_distribution.csv",
    index=False
)

at_risk_premium_churn.to_csv(
    RETENTION_OUTPUT_DIR / "at_risk_premium_churn.csv",
    index=False
)

high_balance_disengaged_table.to_csv(
    RETENTION_OUTPUT_DIR / "high_balance_disengaged_customers.csv",
    index=False
)

retention_priority_distribution.to_csv(
    RETENTION_OUTPUT_DIR / "retention_priority_distribution.csv",
    index=False
)

retention_priority_churn.to_csv(
    RETENTION_OUTPUT_DIR / "retention_priority_churn.csv",
    index=False
)

financial_commitment_kpi_summary.to_csv(
    RETENTION_OUTPUT_DIR / "financial_commitment_kpi_summary.csv",
    index=False
)

financial_commitment_insight_table.to_csv(
    RETENTION_OUTPUT_DIR / "financial_commitment_insight_table.csv",
    index=False
)

print("Financial commitment and retention priority analysis outputs exported successfully.")
print(f"Output Directory: {RETENTION_OUTPUT_DIR}")
print("Files Exported: 12")

Financial commitment and retention priority analysis outputs exported successfully.
Output Directory: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\processed\retention_priority_analysis
Files Exported: 12


## Dashboard Mapping for Notebook 06 Outputs

This section maps Notebook 06 analytical outputs to the future Streamlit dashboard modules.

In [21]:
# Step 6V: Create dashboard mapping for Notebook 06 outputs

financial_commitment_dashboard_mapping = pd.DataFrame({
    "Dashboard Module": [
        "Customer Risk Segment Overview",
        "Risk Segment Churn Analysis",
        "High-Balance Engagement Analysis",
        "Salary-Balance Mismatch Detector",
        "At-Risk Premium Customer Panel",
        "High-Value Disengaged Customer Table",
        "Retention Priority Scoring Panel",
        "Financial Commitment KPI Summary",
        "Financial Commitment Insight Summary"
    ],
    "Notebook Output Used": [
        "customer_risk_distribution.csv",
        "customer_risk_churn.csv",
        "high_balance_engagement_churn.csv",
        "salary_balance_mismatch_churn.csv",
        "at_risk_premium_churn.csv",
        "high_balance_disengaged_customers.csv",
        "retention_priority_churn.csv",
        "financial_commitment_kpi_summary.csv",
        "financial_commitment_insight_table.csv"
    ],
    "Dashboard Purpose": [
        "Show customer distribution across behavior-based risk groups",
        "Compare churn and retention across customer risk segments",
        "Identify whether high balance customers remain engaged or become inactive",
        "Detect customers with high salary but weak balance commitment",
        "Track premium-risk customers who may need proactive retention action",
        "List financially valuable but inactive customers for targeted follow-up",
        "Prioritize customers into critical, high, medium, and low retention segments",
        "Display executive-level financial commitment KPIs",
        "Summarize key findings and dashboard-ready business insights"
    ]
})

financial_commitment_dashboard_mapping

,Dashboard Module,Notebook Output Used,Dashboard Purpose
0,Customer Risk Segment Overview,customer_risk_distribution.csv,Show customer distribution across behavior-bas...
1,Risk Segment Churn Analysis,customer_risk_churn.csv,Compare churn and retention across customer ri...
2,High-Balance Engagement Analysis,high_balance_engagement_churn.csv,Identify whether high balance customers remain...
3,Salary-Balance Mismatch Detector,salary_balance_mismatch_churn.csv,Detect customers with high salary but weak bal...
4,At-Risk Premium Customer Panel,at_risk_premium_churn.csv,Track premium-risk customers who may need proa...
5,High-Value Disengaged Customer Table,high_balance_disengaged_customers.csv,List financially valuable but inactive custome...
6,Retention Priority Scoring Panel,retention_priority_churn.csv,"Prioritize customers into critical, high, medi..."
7,Financial Commitment KPI Summary,financial_commitment_kpi_summary.csv,Display executive-level financial commitment KPIs
8,Financial Commitment Insight Summary,financial_commitment_insight_table.csv,Summarize key findings and dashboard-ready bus...


## Notebook 06 Final Conclusion

Notebook 06 completed the financial commitment, premium risk, and retention priority analysis for the Customer Engagement & Product Utilization Analytics project.

The analysis shows that financial value alone does not guarantee customer retention. High-balance customers can still become retention risks when engagement is weak, product usage is limited, or relationship strength is low.

Key final findings:

- High-balance customer behavior was analyzed using balance percentile logic.
- High-balance disengaged customers were identified as a major retention opportunity.
- Salary-balance mismatch helped detect customers with high earning potential but weak account commitment.
- At-risk premium customers were separated for proactive monitoring.
- Credit card ownership showed only a small retention lift, so it should not be treated as the only stickiness signal.
- Retention priority segments were created to classify customers into actionable retention groups.
- Critical and high-priority customer groups can directly support retention planning and dashboard decision-making.
- Financial commitment should be evaluated together with engagement status, product depth, churn status, and relationship strength.

The exported outputs are now ready to support the Streamlit dashboard module:

`Financial Commitment & Retention Priority Overview`

The next notebook will focus on deeper retention scoring, final segmentation consolidation, or dashboard-ready master dataset preparation.

## Final Checkpoint Confirmation

In [24]:
# Step 6X: Final checkpoint confirmation for Notebook 06

source_df_check = pd.read_csv(FEATURES_DATA_PATH)

print("Notebook 06 completed successfully.")
print("Analysis Area: Financial Commitment, Premium Risk and Retention Priority Analysis")
print(f"Original Feature-Engineered Input Rows: {source_df_check.shape[0]:,}")
print(f"Original Feature-Engineered Input Columns: {source_df_check.shape[1]}")
print(f"Notebook 06 Working DataFrame Columns After Analysis: {df.shape[1]}")
print("New Columns Created in Notebook 06: 3")
print("Output Files Exported: 12")
print("Dashboard Module Ready: Financial Commitment & Retention Priority Overview")

Notebook 06 completed successfully.
Analysis Area: Financial Commitment, Premium Risk and Retention Priority Analysis
Original Feature-Engineered Input Rows: 10,000
Original Feature-Engineered Input Columns: 33
Notebook 06 Working DataFrame Columns After Analysis: 36
New Columns Created in Notebook 06: 3
Output Files Exported: 12
Dashboard Module Ready: Financial Commitment & Retention Priority Overview


In [25]:
# Export final retention priority master dataset for Notebook 07

RETENTION_PRIORITY_MASTER_PATH = RETENTION_OUTPUT_DIR / "retention_priority_master_dataset.csv"

df.to_csv(RETENTION_PRIORITY_MASTER_PATH, index=False)

print("Retention priority master dataset exported successfully.")
print("Output Path:", RETENTION_PRIORITY_MASTER_PATH)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Retention priority master dataset exported successfully.
Output Path: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\processed\retention_priority_analysis\retention_priority_master_dataset.csv
Rows: 10000
Columns: 36
